# CloudGraph API Guide

This notebook demonstrates how to interact with the CloudGraph FastAPI backend, handling photo uploads, async serverless EXIF processing, S3 storage, DynamoDB persistence, and native relationship mapping.

## 1. Setup Configuration
Configure your local variables below. You will need a valid Cognito token to authenticate.

In [ ]:
import requests
import json
import time

# Set your configurations here
API_BASE_URL = "http://localhost:8000/api"

# Place a valid Cognito ID token or Access token here
AUTH_TOKEN = "YOUR_COGNITO_TOKEN_HERE"

# Note: The file should be a valid JPEG, PNG, or HEIC image
IMAGE_PATH = "sample_photo.jpg"

headers = {
    "Authorization": f"Bearer {AUTH_TOKEN}"
}

## 2. Uploading a Photo (Asynchronous Processing)
Upload the image file. The backend will:
1. Validate your Cognito JWT token.
2. Upload the raw image bytes to AWS S3 rapidly.
3. Trigger AWS Lambda workers in the background (via S3 events) to extract EXIF and generate a 300x300 thumbnail.
4. Return the `file_key`, `presigned_url`, and a processing message immediately.

In [ ]:
upload_endpoint = f"{API_BASE_URL}/upload"

try:
    with open(IMAGE_PATH, "rb") as f:
        files = {"file": (IMAGE_PATH, f, "image/jpeg")}
        
        print(f"Uploading {IMAGE_PATH} to {upload_endpoint}...")
        response = requests.post(upload_endpoint, headers=headers, files=files)
        
        response_data = response.json()
        
        if response.status_code == 200:
            print("\n\u2705 Upload Initiated successfully!")
            print(json.dumps(response_data, indent=2))
            presigned_url = response_data.get("presigned_url")
        else:
            print(f"\n\u274c Upload Failed with status code {response.status_code}")
            print(response_data)
            presigned_url = None
            
except FileNotFoundError:
    print(f"\u274c Could not find '{IMAGE_PATH}'.")
    presigned_url = None
except Exception as e:
    print(f"\u274c Request failed: {e}")
    presigned_url = None

## 3. Querying Graph Relationships
You can explicitly map how your photos relate to one another via spatial and chronological graph boundaries securely stored against DynamoDB records.

In [ ]:
graph_endpoint = f"{API_BASE_URL}/graph"

params = {
    "time_threshold_minutes": 120,    # Connect photos shot within 2 hours
    "distance_threshold_km": 5.0      # Connect photos sharing a 5 kilometer perimeter
}

print(f"Fetching D3-ready mapped relationships...")
response = requests.get(graph_endpoint, headers=headers, params=params)

if response.status_code == 200:
    graph_data = response.json()
    print(f"\n\u2705 Found {len(graph_data.get('nodes', []))} Nodes and {len(graph_data.get('edges', []))} Edges.\n")
    print(json.dumps(graph_data, indent=2)[:500] + "\n...[truncated output]")
else:
    print(f"\n\u274c Graph Fetch Failed: HTTP {response.status_code}\n{response.text}\n")


## 4. Querying DBSCAN Clusters (Async Lambda Polling)
You can group your photos into unique named clusters using scikit-learn DBSCAN models. For users with > 50 photos, this endpoint dynamically triggers AWS Lambda execution asynchronously.

In [ ]:
cluster_endpoint = f"{API_BASE_URL}/clusters"

cluster_params = {
    "mode": "combined",
    "time_eps_minutes": 60,
    "distance_eps_km": 1.0,
    "min_samples": 2
}

max_retries = 5
for attempt in range(max_retries):
    print(f"Fetching DBSCAN clusters (Attempt {attempt+1}/{max_retries})...")
    res = requests.get(cluster_endpoint, headers=headers, params=cluster_params)
    
    if res.status_code == 200:
        data = res.json()
        
        # Support for asynchronous workflow
        if data.get("status") == "processing":
            print("\u23f3 Clustering is running in the background via AWS Lambda. Polling in 5 seconds...")
            time.sleep(5)
            continue
            
        print(f"\n\u2705 Clusters resolved!")
        print(f"Found {len(data.get('clusters', []))} Clusters and {len(data.get('unclustered', []))} Unclustered Noise nodes.\n")
        print(json.dumps(data, indent=2)[:800] + "\n...[truncated output]")
        break
    else:
        print(f"\n\u274c Cluster Fetch Failed: HTTP {res.status_code}\n{res.text}")
        break